# 06 — Gold Layer: Star Schema Dimensions
Creates static dimension tables for the Star Schema and persists them to tlc_gold in MongoDB.

In [ ]:
from src.core.spark import get_spark
from src.core.mongo import write_mongo
from src.core.config import settings
from src import paths
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [ ]:
spark = get_spark("TLC_Gold_Dimensions")

In [ ]:
vendor_data = [(1, "Creative Mobile Technologies"), (2, "VeriFone")]
dim_vendor = spark.createDataFrame(vendor_data, ["id", "name"])
write_mongo(dim_vendor, "tlc_gold", "dim_vendor")

In [ ]:
rate_code_data = [(1, "Standard rate"), (2, "JFK"), (3, "Newark"), (4, "Nassau or Westchester"), (5, "Negotiated fare"), (6, "Group ride")]
dim_rate_code = spark.createDataFrame(rate_code_data, ["id", "description"])
write_mongo(dim_rate_code, "tlc_gold", "dim_rate_code")

In [ ]:
payment_type_data = [("Credit Card", "Credit Card"), ("Cash", "Cash"), ("No Charge", "No Charge"), ("Dispute", "Dispute"), ("Unknown", "Unknown"), ("Voided Trip", "Voided Trip")]
dim_payment_type = spark.createDataFrame(payment_type_data, ["id", "name"])
write_mongo(dim_payment_type, "tlc_gold", "dim_payment_type")

In [ ]:
vehicle_data = [("yellow", "Yellow Taxi"), ("green", "Green Taxi"), ("fhv", "For-Hire Vehicle"), ("hvfhv", "High Volume FHV")]
dim_vehicle = spark.createDataFrame(vehicle_data, ["id", "name"])
write_mongo(dim_vehicle, "tlc_gold", "dim_vehicle")

In [ ]:
dim_zone = spark.read.csv(str(paths.TAXI_ZONE_LOOKUP), header=True, inferSchema=True) \
    .select(
        F.col("LocationID").alias("zone_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone_name"),
        F.col("service_zone")
    )
write_mongo(dim_zone, "tlc_gold", "dim_zone")

In [ ]:
dim_date = spark.createDataFrame([(1,)], ["dummy"]) \
    .withColumn("date", F.explode(F.sequence(F.to_date(F.lit('2024-01-01')), F.to_date(F.lit('2025-12-31'))))) \
    .drop("dummy") \
    .select(
        F.date_format(F.col("date"), "yyyyMMdd").alias("date_id"),
        F.col("date"),
        F.year(F.col("date")).alias("year"),
        F.month(F.col("date")).alias("month"),
        F.dayofmonth(F.col("date")).alias("day"),
        F.dayofweek(F.col("date")).alias("day_of_week")
    )
write_mongo(dim_date, "tlc_gold", "dim_date")